# Keyword Tree — Multiple Pattern Matching

In [1]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")
import networkx as nx


def hierarchy_pos(G, root="root", width=2.0, vert_gap=1.0):
    """Position nodes in a top-down tree layout (pure Python, no graphviz)."""
    def _subtree_width(node):
        children = list(G.successors(node))
        if not children:
            return 1
        return sum(_subtree_width(c) for c in children)

    def _assign(node, left, right, depth, pos):
        pos[node] = ((left + right) / 2, -depth * vert_gap)
        children = sorted(G.successors(node))
        if not children:
            return
        total_w = sum(_subtree_width(c) for c in children)
        x = left
        for child in children:
            w = _subtree_width(child)
            child_right = x + (right - left) * w / total_w
            _assign(child, x, child_right, depth + 1, pos)
            x = child_right

    pos = {}
    _assign(root, 0, width, 0, pos)
    return pos

Helpers loaded (retina mode).


## Building the Keyword Tree

Enter a comma-separated list of patterns and see the resulting keyword tree.
The stats panel shows size (number of edges), height (longest pattern),
and maximum fan-out.

In [2]:
def build_keyword_tree(patterns):
    """Build a keyword tree as a networkx DiGraph from a list of patterns."""
    G = nx.DiGraph()
    G.add_node("root", label="", depth=0)
    for idx, P in enumerate(patterns):
        cur = "root"
        for j, ch in enumerate(P):
            nxt = None
            for _, neighbor, data in G.out_edges(cur, data=True):
                if data["label"] == ch:
                    nxt = neighbor
                    break
            if nxt is None:
                name = f"{cur}_{ch}"
                while name in G.nodes:
                    name = name + "\'"
                is_leaf = (j == len(P) - 1)
                node_label = P if is_leaf else ""
                G.add_node(name, label=node_label, is_leaf=is_leaf,
                           depth=j + 1, pattern=P if is_leaf else "")
                G.add_edge(cur, name, label=ch)
                cur = name
            else:
                cur = nxt
                if j == len(P) - 1:
                    G.nodes[cur]["is_leaf"] = True
                    G.nodes[cur]["label"] = P
                    G.nodes[cur]["pattern"] = P
    return G


def draw_keyword_tree(G, patterns, title=""):
    """Draw keyword tree with stats panel."""
    n_edges = G.number_of_edges()
    depths = [G.nodes[n].get("depth", 0) for n in G.nodes]
    height = max(depths) if depths else 0
    fan_outs = [G.out_degree(n) for n in G.nodes]
    max_fan = max(fan_outs) if fan_outs else 0

    fig, (ax, ax_stats) = plt.subplots(
        1, 2, figsize=(max(len(G.nodes) * 0.7, 8), max(height * 1.1, 4)),
        gridspec_kw={"width_ratios": [3, 1]})

    pos = hierarchy_pos(G, root="root", width=max(len(G.nodes) * 0.3, 2.0))

    leaves = [n for n in G.nodes if G.nodes[n].get("is_leaf", False)]
    internals = [n for n in G.nodes if n not in leaves]

    nx.draw_networkx_nodes(G, pos, nodelist=internals, ax=ax,
                           node_color="#BBDEFB", node_size=400, edgecolors="#555")
    nx.draw_networkx_nodes(G, pos, nodelist=leaves, ax=ax,
                           node_color="#FFCC80", node_size=400, edgecolors="#E65100",
                           linewidths=2.0)

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#777",
                           arrows=True, arrowsize=12, width=1.5)

    edge_labels = {(u, v): d["label"] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax,
                                  font_size=12, font_color="#9C27B0",
                                  font_family="monospace")

    for n in leaves:
        x, y = pos[n]
        pat = G.nodes[n].get("pattern", "")
        if pat:
            ax.text(x, y - 0.35, pat, ha="center", va="top",
                    fontsize=9, fontstyle="italic", color="#E65100",
                    fontfamily="monospace")

    rx, ry = pos["root"]
    ax.text(rx, ry + 0.3, "root", ha="center", va="bottom",
            fontsize=9, color="#999")

    ax.set_title(title or "Keyword Tree", fontsize=13, pad=12)
    ax.axis("off")

    ax_stats.axis("off")
    stats_text = (
        f"Patterns:  {len(patterns)}\n\n"
        f"Size (edges):  {n_edges}\n\n"
        f"Height:  {height}\n\n"
        f"Max fan-out:  {max_fan}\n\n"
        f"\u03a3 |P\u1d62| = {sum(len(p) for p in patterns)}"
    )
    ax_stats.text(0.1, 0.7, stats_text, fontsize=12, fontfamily="monospace",
                  va="top", ha="left",
                  bbox=dict(boxstyle="round,pad=0.5", facecolor="#F5F5F5",
                            edgecolor="#CCC"))
    ax_stats.set_title("Stats", fontsize=13, pad=12)

    safe_tight_layout()
    plt.show()


# --- Widgets ---
patterns_input = widgets.Text(
    value="potato, tatoo, theater, other",
    description="Patterns:",
    layout=widgets.Layout(width="500px"),
)
out_kw_build = widgets.Output()


def refresh_kw_build(_=None):
    text = patterns_input.value
    if text:
        pats = [p.strip() for p in text.split(",") if p.strip()]
        if pats and len(pats) <= 20 and sum(len(p) for p in pats) <= 100:
            with out_kw_build:
                clear_output(wait=True)
                G_kw = build_keyword_tree(pats)
                draw_keyword_tree(G_kw, pats,
                                  title=f'K(P) for P = {{{", ".join(pats)}}}')
        else:
            with out_kw_build:
                clear_output(wait=True)
                print("Keep total pattern length <= 100 and at most 20 patterns.")


patterns_input.observe(refresh_kw_build, "value")
display(patterns_input, out_kw_build)
refresh_kw_build()

Text(value='potato, tatoo, theater, other', description='Patterns:', layout=Layout(width='500px'))

Output()

## Searching with the Keyword Tree

Enter a text and patterns. Step through each starting position to see
the descent from the root: active edges light up, non-matching branches
are grayed out.

In [3]:
def keyword_search_trace(T, patterns):
    """Trace the keyword-tree search: one step per starting position s.
    Returns list of (s, path_edges, matched_pattern_or_None)."""
    G = build_keyword_tree(patterns)
    steps = []
    for s in range(len(T)):
        cur = "root"
        path_edges = []
        matched = None
        for j in range(s, len(T)):
            ch = T[j]
            nxt = None
            for _, neighbor, data in G.out_edges(cur, data=True):
                if data["label"] == ch:
                    nxt = neighbor
                    path_edges.append((cur, neighbor))
                    break
            if nxt is None:
                break
            cur = nxt
            if G.nodes[cur].get("is_leaf", False):
                matched = G.nodes[cur].get("pattern", "")
                break
        steps.append((s, path_edges, matched))
    return G, steps


def draw_keyword_search_step(T, G, steps, step_idx):
    """Draw one step of keyword-tree search over T."""
    s, path_edges, matched = steps[step_idx]
    path_edge_set = set(path_edges)
    path_node_set = {"root"}
    for u, v in path_edges:
        path_node_set.add(u)
        path_node_set.add(v)

    matches_so_far = [(st[0], st[2]) for st in steps[:step_idx + 1] if st[2]]

    fig, (ax_text, ax_tree) = plt.subplots(
        2, 1, figsize=(max(len(T) * 0.85, 10), max(len(G.nodes) * 0.4, 5)),
        gridspec_kw={"height_ratios": [1, 3]})

    ax_text.set_xlim(-2.5, len(T) + 1.5)
    ax_text.set_ylim(-0.5, 2.5)
    ax_text.set_aspect("equal")
    ax_text.axis("off")

    for i in range(len(T)):
        ax_text.text(i + 0.45, 2.2, str(i), ha="center", va="center",
                     fontsize=9, color="#999")

    depth_reached = len(path_edges)
    t_hi = {}
    for j in range(len(T)):
        if j == s and depth_reached == 0:
            t_hi[j] = COLORS["mismatch"]
        elif s <= j < s + depth_reached:
            t_hi[j] = COLORS["match"] if matched else COLORS["current"]
        elif j == s + depth_reached and not matched and depth_reached > 0:
            t_hi[j] = COLORS["mismatch"]

    draw_string_row(ax_text, 0.8, T, label="T", highlights=t_hi)

    ax_text.annotate("s=" + str(s), xy=(s + 0.45, 0.8),
                     xytext=(s + 0.45, 0.15), fontsize=10, color="#555",
                     ha="center", fontweight="bold",
                     arrowprops=dict(arrowstyle="->", color="#555", lw=1.5))

    if matched:
        status = f'MATCH: "{matched}" at position {s}'
        ax_text.text(len(T) + 0.5, 1.2, status, fontsize=10,
                     color=COLORS["match"], va="center", fontweight="bold")
    elif depth_reached == 0:
        ax_text.text(len(T) + 0.5, 1.2,
                     f"no edge for '{T[s]}'" if s < len(T) else "",
                     fontsize=10, color=COLORS["mismatch"], va="center")
    else:
        mismatch_pos = s + depth_reached
        if mismatch_pos < len(T):
            ax_text.text(len(T) + 0.5, 1.2,
                         f"stuck at '{T[mismatch_pos]}'",
                         fontsize=10, color=COLORS["mismatch"], va="center")

    ax_text.set_title(
        f"Step {step_idx + 1}/{len(steps)}  |  "
        f"matches found: {len(matches_so_far)}",
        fontsize=11, pad=8)

    pos = hierarchy_pos(G, root="root", width=max(len(G.nodes) * 0.3, 2.0))

    active_internals = [n for n in G.nodes if n in path_node_set
                        and not G.nodes[n].get("is_leaf", False)]
    active_leaves = [n for n in G.nodes if n in path_node_set
                     and G.nodes[n].get("is_leaf", False)]
    dim_internals = [n for n in G.nodes if n not in path_node_set
                     and not G.nodes[n].get("is_leaf", False)]
    dim_leaves = [n for n in G.nodes if n not in path_node_set
                  and G.nodes[n].get("is_leaf", False)]

    if dim_internals:
        nx.draw_networkx_nodes(G, pos, nodelist=dim_internals, ax=ax_tree,
                               node_color="#E8E8E8", node_size=350,
                               edgecolors="#CCC", alpha=0.4)
    if dim_leaves:
        nx.draw_networkx_nodes(G, pos, nodelist=dim_leaves, ax=ax_tree,
                               node_color="#F0E0D0", node_size=350,
                               edgecolors="#CCC", linewidths=1.5, alpha=0.4)

    if active_internals:
        nx.draw_networkx_nodes(G, pos, nodelist=active_internals, ax=ax_tree,
                               node_color="#BBDEFB", node_size=450,
                               edgecolors="#1565C0", linewidths=2.0)
    if active_leaves:
        match_color = "#66BB6A" if matched else "#FFCC80"
        edge_color = "#2E7D32" if matched else "#E65100"
        nx.draw_networkx_nodes(G, pos, nodelist=active_leaves, ax=ax_tree,
                               node_color=match_color, node_size=450,
                               edgecolors=edge_color, linewidths=2.5)

    dim_edges = [(u, v) for u, v in G.edges() if (u, v) not in path_edge_set]
    if dim_edges:
        nx.draw_networkx_edges(G, pos, edgelist=dim_edges, ax=ax_tree,
                               edge_color="#DDD", arrows=True, arrowsize=10,
                               width=1.0, alpha=0.4)

    if path_edges:
        edge_c = COLORS["match"] if matched else "#1565C0"
        nx.draw_networkx_edges(G, pos, edgelist=path_edges, ax=ax_tree,
                               edge_color=edge_c, arrows=True, arrowsize=14,
                               width=3.0)

    active_elabels = {(u, v): d["label"] for u, v, d in G.edges(data=True)
                      if (u, v) in path_edge_set}
    dim_elabels = {(u, v): d["label"] for u, v, d in G.edges(data=True)
                   if (u, v) not in path_edge_set}

    if dim_elabels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=dim_elabels,
                                      ax=ax_tree, font_size=10,
                                      font_color="#CCC",
                                      font_family="monospace")
    if active_elabels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=active_elabels,
                                      ax=ax_tree, font_size=13,
                                      font_color="#9C27B0",
                                      font_family="monospace",
                                      font_weight="bold")

    for n in G.nodes:
        if G.nodes[n].get("is_leaf", False):
            x, y = pos[n]
            pat = G.nodes[n].get("pattern", "")
            if pat:
                alpha = 1.0 if n in path_node_set else 0.35
                ax_tree.text(x, y - 0.35, pat, ha="center", va="top",
                             fontsize=9, fontstyle="italic",
                             color="#E65100", fontfamily="monospace",
                             alpha=alpha)

    rx, ry = pos["root"]
    ax_tree.text(rx, ry + 0.3, "root", ha="center", va="bottom",
                 fontsize=9, color="#999")

    ax_tree.axis("off")
    safe_tight_layout()
    plt.show()


# --- Widgets ---
T_kw = widgets.Text(value="a potato that was put in a theater",
                     description="T:", layout=widgets.Layout(width="500px"))
P_kw = widgets.Text(value="potato, tatoo, theater, other",
                     description="Patterns:", layout=widgets.Layout(width="500px"))
step_kw = widgets.IntSlider(value=0, min=0, max=0, description="Step:",
                             continuous_update=True,
                             layout=widgets.Layout(display="none"))


def _update_kw_max(*_):
    T, pats_str = T_kw.value, P_kw.value
    if T and pats_str:
        pats = [p.strip() for p in pats_str.split(",") if p.strip()]
        if pats:
            step_kw.max = max(len(T) - 1, 0)


T_kw.observe(_update_kw_max, "value")
P_kw.observe(_update_kw_max, "value")
_update_kw_max()


def _draw_kw_search(T, pats_str, step):
    if T and pats_str:
        pats = [p.strip() for p in pats_str.split(",") if p.strip()]
        if pats and len(T) > 0:
            G_kw, steps_kw = keyword_search_trace(T, pats)
            draw_keyword_search_step(T, G_kw, steps_kw, step)


out_kw_search = widgets.interactive_output(
    _draw_kw_search, {"T": T_kw, "pats_str": P_kw, "step": step_kw})
stepper_kw = make_stepper(step_kw, "Step")
display(T_kw, P_kw, stepper_kw, out_kw_search)

Text(value='a potato that was put in a theater', description='T:', layout=Layout(width='500px'))

Text(value='potato, tatoo, theater, other', description='Patterns:', layout=Layout(width='500px'))

Output()